# REA Pipeline Notebook

Flow:
1. ReaRequirementsExtractor
2. ReaDeonticStagePipeline
3. VectorEmbedding_v2 (using main REG constraints)
4. Reranker
5. Generate prompt
6. Send prompt (`False` by default)
7. ReadableLlmResponse


In [ ]:
from pathlib import Path
import json
import os
import sys
from dotenv import load_dotenv

project_root = Path("/Users/my/Documents/projects/detectionDeviation").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pipeline.extractor.ReaRequirementsExtractor import ReaRequirementsExtractor
from pipeline.reasoning.ReaDeonticStagePipeline import ReaDeonticStagePipeline
from pipeline.retrieval.VectorEmbedding_v2 import VectorSearch
from pipeline.retrieval.Reranker import Reranker
from pipeline.reasoning.SingleReaStep2To4Pipeline import SingleReaStep2To4Pipeline
from pipeline.reasoning.SendPrompt import SendPrompt
from pipeline.reasoning.ReadableLlmResponse import ReadableLlmResponse

print("Project root:", project_root)


In [ ]:
# -------------------------------
# Config
# -------------------------------
REA_INPUT_TXT_DIR = project_root / "input" / "rea_case3_injections" / "chunked_texts"
REA_REQUIREMENTS_ROOT = project_root / "intermediate_results" / "rea_case3_injections"
REA_DEONTIC_ROOT = project_root / "intermediate_results" / "rea_case3_injections_deontic_stages"

REG_MAIN_SLOTS_JSON = project_root / "intermediate_results" / "reg_eu_ai_act" / "eu_ai_requirements_slots_main.json"
CHROMA_PERSIST_DIR = project_root / "pipeline" / "retrieval" / "chromadb_store"

ARTIFACT_01_DIR = project_root / "intermediate_outputs" / "artifact_01_case3_v3"
ARTIFACT_01_RERANKED_DIR = project_root / "intermediate_outputs" / "artifact_01_case3_v3_reranked"
PROMPT_ROOT_DIR = project_root / "intermediate_outputs" / "single_rea_step2_4"

REASONING_ENV_PATH = project_root / "pipeline" / "reasoning" / ".env"
OPENAI_MODEL = "gpt-5"

VECTOR_MODEL = "BAAI/bge-large-en-v1.5"
VECTOR_COLLECTION_NAME = "gdpr_requirements"
VECTOR_TOP_K = 100
RERANK_TOP_N = 100
PROMPT_TOP_K = 3

REBUILD_REG_INDEX = True
SEND_PROMPTS = False  # default: do not send prompts

RUN_REQUIREMENTS_EXTRACTOR = True
RUN_DEONTIC_STAGE_PIPELINE = True
RUN_VECTOR_EMBEDDING = True
RUN_RERANKER = True
RUN_PROMPT_GENERATION = True
RUN_READABLE_RESPONSE = True

print("Config ready")
print("SEND_PROMPTS:", SEND_PROMPTS)


In [ ]:
# Step 1: REA requirements extraction
if RUN_REQUIREMENTS_EXTRACTOR:
    saved_paths = ReaRequirementsExtractor.process_folder(
        input_folder=REA_INPUT_TXT_DIR,
        output_root_folder=REA_REQUIREMENTS_ROOT,
    )
    print(f"Saved {len(saved_paths)} requirement files")
else:
    print("Skipped Step 1")


In [ ]:
# Step 2: REA deontic stages (stage1-4)
if RUN_DEONTIC_STAGE_PIPELINE:
    stage_runner = ReaDeonticStagePipeline(
        endpoint_url="http://localhost:11434/api/chat",
        model_name="llama3",
        timeout_seconds=240,
        max_retries=3,
        temperature=0.1,
    )
    stage_report = stage_runner.run_folder(
        input_root=REA_REQUIREMENTS_ROOT,
        output_root=REA_DEONTIC_ROOT,
    )
    print("Saved:", stage_report)
else:
    print("Skipped Step 2")


In [ ]:
# Step 3: Vector embedding + search using main REG constraints
if RUN_VECTOR_EMBEDDING:
    search = VectorSearch(
        model_name=VECTOR_MODEL,
        collection_name=VECTOR_COLLECTION_NAME,
    )
    if REBUILD_REG_INDEX:
        embed_result = search.embed_and_store(
            input_json_path=REG_MAIN_SLOTS_JSON,
            chroma_persist_dir=CHROMA_PERSIST_DIR,
            batch_size=32,
        )
        print("Index build:", json.dumps(embed_result, indent=2, ensure_ascii=False))
    else:
        print("Index build skipped")

    search_result = search.vector_search_for_rea_folder(
        rea_json_root_path=REA_DEONTIC_ROOT,
        chroma_persist_dir=CHROMA_PERSIST_DIR,
        artifact_root_dir=ARTIFACT_01_DIR,
        top_k=VECTOR_TOP_K,
    )
    print("Vector search done. Chunks:", search_result.get("chunk_count"))
else:
    print("Skipped Step 3")


In [ ]:
# Step 4: Reranking
if RUN_RERANKER:
    load_dotenv(REASONING_ENV_PATH)
    reranker = Reranker(
        model_name="kanon-2-reranker",
        api_key_env="ISAACUS_API_KEY",
    )
    rerank_result = reranker.rerank_artifact_root(
        input_artifact_01_dir=ARTIFACT_01_DIR,
        output_artifact_dir=ARTIFACT_01_RERANKED_DIR,
        top_n=RERANK_TOP_N,
    )
    print("Rerank done. Chunks:", rerank_result.get("chunk_count"))
else:
    print("Skipped Step 4")


In [ ]:
# Step 5: Generate prompt payloads (step2-4 prompt pipeline)
if RUN_PROMPT_GENERATION:
    prompt_pipeline = SingleReaStep2To4Pipeline(project_root=project_root)
    prompt_report = prompt_pipeline.run_folder(
        reranked_root=ARTIFACT_01_RERANKED_DIR,
        output_root=PROMPT_ROOT_DIR,
        top_k=PROMPT_TOP_K,
    )
    print("Saved prompt report:", prompt_report)
else:
    print("Skipped Step 5")


In [ ]:
# Step 6: Send prompts (disabled by default)
if SEND_PROMPTS:
    sender = SendPrompt(
        env_path=REASONING_ENV_PATH,
        model_name=OPENAI_MODEL,
    )
    prompt_files = sorted(PROMPT_ROOT_DIR.rglob("step4_prompt_payload.json"))
    sent_outputs = []
    for prompt_file in prompt_files:
        saved = sender.send_prompt_json(prompt_json_path=prompt_file)
        sent_outputs.append(str(saved))
    print(f"Sent {len(sent_outputs)} prompts")
else:
    print("Skipped Step 6 (SEND_PROMPTS=False)")


In [ ]:
# Step 7: Readable LLM responses
if RUN_READABLE_RESPONSE:
    formatter = ReadableLlmResponse()
    response_files = sorted(PROMPT_ROOT_DIR.rglob("*_llm_response.json"))
    readable_outputs = []
    for response_file in response_files:
        saved = formatter.run(llm_response_json=response_file)
        readable_outputs.append(saved)
    print(f"Formatted {len(readable_outputs)} response files")
else:
    print("Skipped Step 7")
